# grokRegister-cpa · Colab 一键注册

使用仓库自带 CLI（`python grok_register_ttk.py cli start`），**不重写注册逻辑**。

**限制（必读）**
- 协议模式 Turnstile 需要 Chrome；默认 **有头 + Xvfb**（真 headless 易被 CF 拦）
- Colab 出口多为 Google 机房 IP，可能被 Cloudflare `Attention Required`
- 不要用云盘；上传 notebook / clone GitHub 即可

从上到下运行每个单元格。

## 0. 修正工作目录

若之前 `rm -rf` 过当前目录，先执行本格，避免 `getcwd` 报错。

In [ ]:
%cd /content
import os
print("cwd =", os.getcwd())

## 1. 安装 Chrome + Xvfb + Python 依赖

In [ ]:
%%bash
cd /content || true
set +e
export DEBIAN_FRONTEND=noninteractive

apt-get update -qq || true
apt-get install -y -qq xvfb fonts-liberation \
  libnss3 libatk-bridge2.0-0 libgtk-3-0 libx11-xcb1 libxcomposite1 libxdamage1 \
  libxrandr2 libgbm1 libasound2 libpangocairo-1.0-0 libcups2 xdg-utils || true

if ! command -v google-chrome-stable >/dev/null 2>&1 && ! command -v google-chrome >/dev/null 2>&1; then
  echo "installing google-chrome-stable..."
  wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
  apt-get install -y -qq /tmp/chrome.deb || dpkg -i /tmp/chrome.deb || true
  apt-get install -y -f -qq || true
fi

python -m pip install -q -U pip

echo "=== checks ==="
which Xvfb || true
which google-chrome-stable || which google-chrome || true
google-chrome-stable --version 2>/dev/null || google-chrome --version 2>/dev/null || true
echo "deps base ok"

## 2. 获取源码

把 `REPO_URL` 改成你的 fork（需含 `colab/` 与最新 CLI）。

In [ ]:
import os
import shutil
from pathlib import Path

os.chdir("/content")

# ====== 按需修改 ======
REPO_URL = "https://github.com/Git-creat7/grokRegister-cpa.git"  # 或你的 fork
BRANCH = "main"
WORK = Path("/content/grokRegister-cpa")
# =====================

if WORK.exists() and not (WORK / "grok_register_ttk.py").is_file():
    print("半拉子目录，删除重 clone")
    shutil.rmtree(WORK, ignore_errors=True)

if not (WORK / "grok_register_ttk.py").is_file():
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {WORK}
else:
    print("repo already present:", WORK)

os.chdir(WORK)
print("cwd =", os.getcwd())
for must in ("grok_register_ttk.py", "requirements.txt", "config.example.json"):
    ok = Path(must).is_file()
    print(("OK" if ok else "MISSING"), must)
    assert ok, f"缺少 {must}，请检查 REPO_URL / BRANCH"

!pip install -q -r requirements.txt
print("pip ok")

## 3. 放入 config.json

任选：
1. 左侧上传到 `/content/grokRegister-cpa/config.json`
2. 或取消注释下方 `CONFIG_JSON` 写入

In [ ]:
from pathlib import Path
import json

cfg_path = Path("/content/grokRegister-cpa/config.json")

# CONFIG_JSON = r'''
# {
#   "email_provider": "cloudflare",
#   "register_count": 1,
#   "register_mode": "protocol",
#   "cpa_auto_add": true,
#   "cpa_auth_dir": "./auth",
#   "chenyme_grok2api_enabled": false,
#   "chenyme_grok2api_base": "https://grok2api.example.com",
#   "chenyme_grok2api_username": "admin",
#   "chenyme_grok2api_password": "",
#   "g2a_build_import_file_enabled": true,
#   "g2a_build_import_file": "exports/grok2api_build_import.json",
#   "g2a_build_remote_import_enabled": false
# }
# '''
# cfg_path.write_text(CONFIG_JSON.strip() + "\n", encoding="utf-8")

if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    print("loaded config.json, keys=", len(cfg))
    print("email_provider=", cfg.get("email_provider"))
    print("register_count=", cfg.get("register_count"))
    print("cpa_auto_add=", cfg.get("cpa_auto_add"))
    print("chenyme_enabled=", cfg.get("chenyme_grok2api_enabled"))
    print("g2a_file=", cfg.get("g2a_build_import_file_enabled"))
    print("g2a_remote=", cfg.get("g2a_build_remote_import_enabled"))
else:
    print("尚未找到 config.json — 请上传到", cfg_path)
    print("或取消注释 CONFIG_JSON 写入逻辑")

## 4. 查看出口 IP

In [ ]:
import json
from curl_cffi import requests

r = requests.get("https://ipinfo.io/json", timeout=15, proxies={})
info = r.json()
print(json.dumps(info, indent=2, ensure_ascii=False))
org = str(info.get("org") or "").lower()
hosting = any(x in org for x in ("google", "cloud", "amazon", "microsoft", "hosting", "server"))
print("hosting_like=", hosting)
if hosting:
    print("提示: 机房 IP 更容易被 Cloudflare 拦；可 Runtime→Disconnect and delete runtime 换机")

## 5. 启动 Xvfb + 注册（CLI auto start）

等价于：`python grok_register_ttk.py cli start`  
默认 **非真 headless**（后台有界面浏览器 + 虚拟显示）。

In [ ]:
import os
import subprocess
import time
from pathlib import Path

WORK = Path("/content/grokRegister-cpa")
os.chdir(WORK)
assert (WORK / "config.json").is_file(), "请先准备 config.json"
assert (WORK / "grok_register_ttk.py").is_file()

# Xvfb 虚拟显示（有头 Chrome 过 CF 比真 headless 靠谱）
if not os.environ.get("DISPLAY"):
    log = open("/tmp/xvfb-cpa.log", "ab")
    subprocess.Popen(
        ["Xvfb", ":99", "-screen", "0", "1280x900x24", "-ac"],
        stdout=log,
        stderr=log,
        start_new_session=True,
    )
    time.sleep(0.8)
    os.environ["DISPLAY"] = ":99"
print("DISPLAY =", os.environ.get("DISPLAY"))

# 可选：覆盖本批数量
# import json
# cfg = json.loads((WORK / "config.json").read_text(encoding="utf-8"))
# cfg["register_count"] = 1
# (WORK / "config.json").write_text(json.dumps(cfg, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

!cd /content/grokRegister-cpa && DISPLAY=:99 python grok_register_ttk.py cli start

## 6. 打包下载结果

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

root = Path("/content/grokRegister-cpa")
out_dir = Path("/content/cpa_colab_out")
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir()

for pat in ("accounts_*.txt", "sso_pending.txt", "nsfw_pending.txt"):
    for p in root.glob(pat):
        shutil.copy2(p, out_dir / p.name)
        print("+", p.name)

for sub in ("exports", "auth", "log"):
    src = root / sub
    if src.is_dir():
        shutil.copytree(src, out_dir / sub, dirs_exist_ok=True)
        print("+", sub + "/")

zip_path = "/content/grokRegister_cpa_colab_out"
shutil.make_archive(zip_path, "zip", out_dir)
files.download(zip_path + ".zip")
print("downloaded", zip_path + ".zip")

## 7. 换机（新 IP）

菜单：**Runtime → Disconnect and delete runtime** → 重新连接 → Run all。

或：

In [ ]:
DO_UNASSIGN = False  # 改 True 再运行（会断开当前会话）

if DO_UNASSIGN:
    from google.colab import runtime
    print("unassign… 请稍后重新连接")
    runtime.unassign()
else:
    print("未执行。把 DO_UNASSIGN=True 再跑，或用菜单 Disconnect and delete runtime")